# Notebook 03 — Diseño Experimental para Imputación de Precipitación Diaria

**TFM:** Evaluación de la influencia del componente espacial en la estimación de datos faltantes de precipitación diaria mediante modelos de Machine Learning aplicados a estaciones meteorológicas.

## 1 - Objetivo del notebook


---


Este notebook **NO entrena los modelos finales**.


El objetivo del presente Notebook es trabajar con las observaciones obtenidas del **Notebook 2**. La idea fundamental es trabajar con observaciones cuyo valor real conocemos, ocultar artificialmente una parte de ellas y guardar por separado para los set de datos para entrenamiento y validación de los modelos los cuales serán usados en el

```text
Notebook 04 — Imputación, modelado y evaluación de precipitación diaria.
```



### 1.1 - Flujo de este notebook

1. Salidas del Notebook 02

2. Comprobar periodo y estaciones

3. Identificar NaN naturales

4. Definir observaciones elegibles

5. Separar train / valid / test

6. Caracterizar días secos y lluviosos

7. Seleccionar observaciones que se ocultarán artificialmente

8. Guardar el valor real como y_true

9. Sustituir temporalmente esas observaciones por NaN

10. Comprobar que el experimento quedó bien construido

11. Guardar dataset experimental

12. Notebook 04: modelado y evaluación

### 1.2 - Importar librerías y definir carpetas del proyecto

La estructura de carpetas esperada es:

```text
TFM_precipitacion/
├── data/
│   ├── raw/
│   ├── interim/
│   └── processed/
├── outputs/
│   ├── tables/
│   ├── figures/
│   └── logs/
└── notebooks/
```

A continuación, descargamos las librerias requeridas para el desarrollo del presente Notebook

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

A continuación se verifica la estructura de archivos que ha implementado para el proyecto, esto es solo una pequeña verificación

In [ ]:

carpeta_actual = Path.cwd()

if carpeta_actual.name == 'notebooks':
    PROJECT_ROOT = carpeta_actual.parent
else:
    PROJECT_ROOT = carpeta_actual

folders = [
    PROJECT_ROOT / "data" / "raw"/ "precipitacion",
    PROJECT_ROOT / "data" / "intermediate",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "notebooks"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

### 1.3 - Entrada archivos - Notebook 2

En este numeral se llaman los archivos generados en el notebook 2

 >* *RESUMEN_PATH*: Corresponde al número y porcentaje anual de días en los que las tres estaciones de la configuración seleccionada tienen observaciones simultáneas.
 >
 >* *PERIODO_SELECCIONADO_PATH*: Corresponde a la df_wide de las tres estaciones seleccionadas para el periodo 2004–2015. Se aclara que este contiene todos los días del calendario y sus NaN naturales.
 >
 >* *ESTACIONES_SELECCIONADAS_PATH*: Contiene la información geográfia de las estaciones seleccionadas. (Componente espacial = ubicación)



In [ ]:
# ArchivoS generadoS por el Notebook 02

RESUMEN_PATH =  PROJECT_ROOT / "data" / "intermediate" / "disponibilidad_periodo_seleccionado.parquet"
PERIODO_SELECCIONADO_PATH =  PROJECT_ROOT / "data" / "intermediate" / "precipitacion_periodo_seleccionado.parquet"
ESTACIONES_SELECCIONADAS_PATH =  PROJECT_ROOT / "data" / "intermediate" / "estaciones_seleccionadas.geojson"


In [ ]:
df_resumen = pd.read_parquet(RESUMEN_PATH)
df_resumen

,Anio,Dias_simultaneos,Dias_anio,Disponibilidad_pct,Configuracion
4,2004,121,366,33.06,A
5,2005,136,365,37.26,A
6,2006,142,365,38.90,A
7,2007,70,365,19.18,A
8,2008,142,366,38.80,A
9,2009,154,365,42.19,A
10,2010,188,365,51.51,A
11,2011,189,365,51.78,A
12,2012,165,366,45.08,A
13,2013,151,365,41.37,A


En la tabla anterior se evidencia el rango de fechas seleccionado en el Notebook anterior, el año con el menor número de registros simultaneos entre las tres estaciones es el 2007. El periodo de años entre el 2004 al 2008 no llegan con una simultaneidad superior a 40%.

In [ ]:
df_periodo_seleccionado = pd.read_parquet(PERIODO_SELECCIONADO_PATH)
df_periodo_seleccionado

CodigoEstacion,21205710,21201230,21201600
Fecha,,,
2004-01-01,NaN,NaN,NaN
2004-01-02,NaN,0.0,NaN
2004-01-03,NaN,NaN,NaN
2004-01-04,NaN,NaN,NaN
2004-01-05,NaN,2.5,0.0
...,...,...,...
2015-12-27,NaN,0.0,NaN
2015-12-28,NaN,0.0,NaN
2015-12-29,NaN,0.0,NaN


Importamos las estaciones seleccionadas como GeoJson para no perder el componente espacial, en este caso su geometría **geometry**

In [ ]:
df_estaciones = gpd.read_file(ESTACIONES_SELECCIONADAS_PATH)
df_estaciones

,CODIGO,nombre,CATEGORIA,TECNOLOGIA,ESTADO,FECHA_INST,altitud,latitud,longitud,DEPARTAMEN,...,d_AREA_HID,d_ZONA_HID,d_CORRIENT,d_SUBZONA_,distancia_m,distancia_km,FECHA_INST_MOD,FECHA_SUSP_MOD,Objetivo,geometry
0,21205710,JARDIN BOTANICO [21205710],CP,"TEST001,TEST003",ESTA001,1974-09-15,2552.0,4.667867,-74.101034,11.0,...,Magdalena Cauca,Alto Magdalena,None,Río Bogotá,2709.989675,2.709990,1974-09-15,NaT,True,POINT (4877927.545 2073889.531)
1,21201230,ENMANUEL D' ALZON [21201230],PM,TEST003,ESTA002,1974-04-15,2520.0,4.701125,-74.070306,11.0,...,Magdalena Cauca,Alto Magdalena,None,Río Bogotá,4167.428588,4.167429,1974-04-15,2025-11-24,False,POINT (4881340.437 2077559.696)
2,21201600,SEDE IDEAM KRA 10 [21201600],PG,TEST003,ESTA002,1986-09-14,2685.0,4.607111,-74.072889,11.0,...,Magdalena Cauca,Alto Magdalena,None,Río Bogotá,6302.648333,6.302648,1986-09-14,2019-05-03,False,POINT (4881038.215 2067170.438)


## 2 - Configuración del experimento


---



Para el experimento se definen las siguientes variables:
* Estación Objetivo
* Estaciones auxiliares
* Fechas de inicio y de Fin de los registros


In [ ]:
ESTACION_OBJETIVO = df_estaciones.loc[df_estaciones["Objetivo"] == True,'CODIGO'].iloc[0]
ESTACION_OBJETIVO

'21205710'

In [ ]:
ESTACIONES_AUXILIARES = df_estaciones.loc[df_estaciones["Objetivo"] == False,'CODIGO'].to_list()
ESTACIONES_AUXILIARES



['21201230', '21201600']

In [ ]:
FECHA_INICIO = df_periodo_seleccionado.index.min()
FECHA_FIN    = df_periodo_seleccionado.index.max()

In [ ]:
print(f"La estación objetivo es: {ESTACION_OBJETIVO}\n")

print(f"Las estaciones auxiliares son: ")
for estacion in ESTACIONES_AUXILIARES:
    print(f"  - {estacion}")


print(f"\nFecha de inicio: {FECHA_INICIO}")
print(f"Fecha final : {FECHA_FIN}")


La estación objetivo es: 21205710

Las estaciones auxiliares son: 
  - 21201230
  - 21201600

Fecha de inicio: 2004-01-01 00:00:00
Fecha final : 2015-12-31 00:00:00


## 3 - Comprobación del conjunto de estudio


---



Si bien en el notebook anterior se depuró la información, en este ítem se hace una verificación básica a la información recibida.

La primer comprobación es la presencia de fechas o registros duplicados.

In [ ]:
df_periodo_seleccionado.index.duplicated().sum()

np.int64(0)

Se hace la comprobación de "Valores Nulos Narutales"  

In [ ]:
for estacion in df_estaciones.CODIGO:
    nulos = df_periodo_seleccionado[estacion].isna().sum()
    print(f"La estación {estacion} tiene un total de {nulos} valores nulos")

La estación 21205710 tiene un total de 690 valores nulos
La estación 21201230 tiene un total de 1499 valores nulos
La estación 21201600 tiene un total de 1915 valores nulos


A continuación, se resume la cantidad de observaciones esperadas y se contrastan con las observaciones medidas para cada una de las estaciones.

In [ ]:
nulos = df_periodo_seleccionado.isna().sum()
observados = df_periodo_seleccionado.notna().sum()

df_disponibilidad = pd.DataFrame({
    "Datos_nulos": nulos,
    "Observados": observados,
    "Disponibilidad_pct": round(observados / len(df_periodo_seleccionado) * 100,2)
})

df_disponibilidad

,Datos_nulos,Observados,Disponibilidad_pct
CodigoEstacion,,,
21205710,690,3693,84.26
21201230,1499,2884,65.80
21201600,1915,2468,56.31


Como se ve en la tabla anterior, la estación **21205710** presenta la mayor disponibilidad de observaciones, esto favorece la disponibilidad de registros reales sobre los datos faltantes para el ejercicio de imputación de datos.


## 4 - Identificación de `NAN` Naturales


---


Dado que el propósito del TFM es determinar la precisión del modelo al estimar datos ausentes, las métricas de evaluación de la calidad de estos necesitan una comparación con los valores conocidos que se ocultaron intencionalmente durante la fase de entrenamiento del modelo. Por lo tanto, en el caso de contar con valores desconocidos naturales, estos pueden ser estimados, pero no evaluados, lo que hace necesario un etiquetado para excluirlos de la evaluación del modelo.

A continuación, se etiquetarán los valores faltantes tanto para la estación objetivo `es_nan_natural_objetivo` como para las estaciones auxiliares `es_nan_natural_aux1` o `es_nan_natural_aux2`

In [ ]:
df_periodo_seleccionado["es_nan_nat_obj"] = df_periodo_seleccionado[ESTACION_OBJETIVO].isna()
df_periodo_seleccionado["es_nan_nat_aux1"] = df_periodo_seleccionado[ESTACIONES_AUXILIARES[0]].isna()
df_periodo_seleccionado["es_nan_nat_aux2"] = df_periodo_seleccionado[ESTACIONES_AUXILIARES[1]].isna()
df_periodo_seleccionado

CodigoEstacion,21205710,21201230,21201600,es_nan_nat_obj,es_nan_nat_aux1,es_nan_nat_aux2
Fecha,,,,,,
2004-01-01,NaN,NaN,NaN,True,True,True
2004-01-02,NaN,0.0,NaN,True,False,True
2004-01-03,NaN,NaN,NaN,True,True,True
2004-01-04,NaN,NaN,NaN,True,True,True
2004-01-05,NaN,2.5,0.0,True,False,False
...,...,...,...,...,...,...
2015-12-27,NaN,0.0,NaN,True,False,True
2015-12-28,NaN,0.0,NaN,True,False,True
2015-12-29,NaN,0.0,NaN,True,False,True


### 4.1 - Comparativa de Tratamiento de Datos Faltantes

#### 1. Día con NaN Natural *(Estación Objetivo)*
> Ocurre cuando los datos faltan de origen en el registro real.

| Acción | Permitido | Notas |
| :--- | :---: | :--- |
| **Imputación** | 🟢 | Se puede imputar finalmente para completar la serie. |
| **Evaluación** | ❌ | **NO** se puede evaluar (no existe un valor real de control para comparar). |




#### 2. Día Artificialmente Ocultado *(Valid/Test)*
> Ocurre cuando se oculta intencionadamente un dato real para probar el modelo.

| Acción | Requisito | Notas |
| :--- | :---: | :--- |
| **Imputación** | 🟢 | Se imputa utilizando el algoritmo a prueba. |
| **Evaluación** | 🟢 | **DEBE** evaluarse contrastando el valor imputado con el valor real oculto. |



## 5 - Definición de Observaciones elegibles


---

En este capítulo se determina cuales son los registros o fechas elegibles para el experimento

In [ ]:
tres_estaciones_simultaneas = df_resumen.Dias_simultaneos.sum() # Número de días con observaciones simultaneas para las 3 estaciones



# Número de días con observaciones simultaneas en la estación objetivo y una de las dos auxiliares

objet_con_data = df_periodo_seleccionado[ESTACION_OBJETIVO].notna()
aux1_con_data = df_periodo_seleccionado[ESTACIONES_AUXILIARES[0]].notna()
aux2_con_data = df_periodo_seleccionado[ESTACIONES_AUXILIARES[1]].notna()

# Caso 1, aux1 con registro y aux2 sin registro
caso1 = objet_con_data & aux1_con_data & ~aux2_con_data

# Caso 2, aux1 sin registro y aux2 con registro
caso2 = objet_con_data & ~aux1_con_data & aux2_con_data


dos_estaciones_simultaneas = (caso1|caso2).sum()


print(f"Los días en los que las estaciones tienen 3 registros simultaneos son {tres_estaciones_simultaneas}")
print(f"Los días en los que las estaciones tienen 2 registros simultaneos son {dos_estaciones_simultaneas}")

Los días en los que las estaciones tienen 3 registros simultaneos son 1789
Los días en los que las estaciones tienen 2 registros simultaneos son 813


### 5.1 - Criterio de Elegibilidad ($Elegible_t$)

Como resultado en el Notebook 02 se registraron 1.789 días con observaciones para en las tres estaciones en el período de (2004–2015). Estos datos corresponden a un volumen importante de información compartida entre la estación objetivo y las auxiliares, lo cual es muy importante en la fase de modelado, pero se debe considerar que hay un grupo adicional que corresponde a 813 registros en el mismo periodo, en donde hay información de la estación objetivo y alguna de las auxiliares.

Por lo anterior, se considera como criterio de elegibilidad únicamente los días en los que las tres estaciones presentan información simultánea resultaría innecesariamente restrictivo, por lo que para el experimento se usaran los dos grupos resumidos anteriormemnte de la siguiente manera.

+ Condicion uno **Objetivo + 1 auxiliar**
$$(P_{obj,t} \neq \text{NaN}) \land ((P_{aux1,t} \neq \text{NaN}) \lor (P_{aux2,t} \neq \text{NaN}))$$

+ Condicion uno **Objetivo + las 2 auxiliares**
$$(P_{obj,t} \neq \text{NaN}) \land (P_{aux1,t} \neq \text{NaN}) \land (P_{aux2,t} \neq \text{NaN})$$





---




| Condición | Cantidad |
| :--- | :---: |
| **Objetivo + exactamente 1 auxiliar** | 813 |
| **Objetivo + las 2 auxiliares** | 1789 |
| **Total elegible** | ≈ 2602 |





---



Resultado: aplicando este criterio se obtienen 2.602 observaciones elegibles. De ellas, 1.789 días cuentan con información de las dos estaciones auxiliares y 813 días disponen únicamente de una de ellas. Esto amplía el conjunto potencial de evaluación sin modificar los valores faltantes naturales ni introducir información artificial.

### 5.2 - Número de estaciones Aux. / día


El criterio principal para la determinación de las fechas elegibles es el de dato conocido para la estación objetivo y uno para alguna de las estaciones auxiliares, por lo que antes de construir la variable `Elegible`, se cuenta cuántas auxiliares tienen observación en cada fecha.


A continuación, se cuenta fila por fila el número de datos de las estaciones auxiliares que cuentan con un valor registrado dando como posible resultado `0`, `1` o `2`.

In [ ]:
df_periodo_seleccionado["n_aux_data"] = (
    df_periodo_seleccionado[ESTACIONES_AUXILIARES]
    .notna()
    .sum(axis=1)
)

df_periodo_seleccionado["n_aux_data"].value_counts().sort_index()

,count
n_aux_data,
0,1151
1,1112
2,2120


### 5.3 - Construcción de las máscaras de elegibilidad

Se crearán dos indicadores:

- `Elegible`: Estacion objetivo conocida y **al menos una** auxiliar disponible. Este será el criterio principal.
- `Completo_3_estaciones`: Estación objetivo y las dos auxiliares conocidas. Permite conservar la referencia de los 1.789 días identificados en el Notebook 02.

In [ ]:
# Criterio principal del la máscara
df_periodo_seleccionado["Elegible"] = (
    ~df_periodo_seleccionado["es_nan_nat_obj"]
    &
    (df_periodo_seleccionado["n_aux_data"] >= 1)
)

# Subconjunto de control: las tres estaciones tienen observación
df_periodo_seleccionado["Completo_3_estaciones"] = (
    ~df_periodo_seleccionado["es_nan_nat_obj"]
    &
    (df_periodo_seleccionado["n_aux_data"] == 2)
)

print("Días elegibles (objetivo + al menos una auxiliar):",
      df_periodo_seleccionado["Elegible"].sum())

print("Días con las tres estaciones completas:",
      df_periodo_seleccionado["Completo_3_estaciones"].sum())

Días elegibles (objetivo + al menos una auxiliar): 2602
Días con las tres estaciones completas: 1789


Con lo anterior se comprueba el criterio de selección para los días elegibles ya que se cuenta con lo siguiente:

- aproximadamente **2.602 días elegibles** con la nueva condición;
- **1.789 días completos** con las tres estaciones conocidas.




el total de 2602 días elegibles robustece el número de registros para la ejecución de los modelos, y como control, se agrega la columna de `Completo_3_estaciones` con la que se puede comprobar el núemero de registros que se concluyeron en el Notebook 2 para la selección de las estaciones.

Los `NaN` naturales **no se modifican** en esta etapa.

## 6 - División temporal: train / validation / test


---

Para una serie temporal, mezclar aleatoriamente registros con fechas aleatorias dentro de los grupos (train-validation-test) podría producir estimaciones demasiado optimistas. La propia documentación de [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html) recomienda divisiones respetando el orden temporal para este tipo de datos.

La separación debe respetar el orden temporal porque los datos pertenecen a una serie diaria, por lo tanto se utiliza:

```text
TRAIN       2004–2011
VALID       2012–2013
TEST        2014–2015
```

Cada bloque contiene años completos y se cumple:

\[
train < valid < test
\]

**Función de cada conjunto**

- **Train:** datos utilizados para entrenar.
- **Valid:** datos destinados a seleccionar o comprobar decisiones de modelado sin utilizar el test.
- **Test:** evaluación final de los métodos; no debe intervenir en decisiones de entrenamiento o ajuste.

La separación se realiza sobre el calendario completo y posteriormente se consultan los días que además cumplen `Elegible`.



In [ ]:

# División temporal - Entrenamiento
TRAIN_INICIO = FECHA_INICIO
TRAIN_FIN = pd.Timestamp("2011-12-31")

# División temporal - Validación
VALID_INICIO = pd.Timestamp("2012-01-01")
VALID_FIN = pd.Timestamp("2013-12-31")

# División temporal - Test
TEST_INICIO = pd.Timestamp("2014-01-01")
TEST_FIN = FECHA_FIN



In [ ]:
df_periodo_seleccionado["split"] = pd.NA

### 6.1 - Asignación de Etiquetas

Por medio de condicionales se crean los **Split** o los grupos de datos para los posteriores trabajos con los modelos.

In [ ]:
condicion_train = (df_periodo_seleccionado.index >= TRAIN_INICIO) & (df_periodo_seleccionado.index <= TRAIN_FIN)
condicion_validacion = (df_periodo_seleccionado.index >= VALID_INICIO) & (df_periodo_seleccionado.index <= VALID_FIN)
condicion_test = (df_periodo_seleccionado.index >= TEST_INICIO) & (df_periodo_seleccionado.index <= TEST_FIN)

Una vez definidas la condiciones de los **splits** se a partir de estos condicionales las respectivas etiquetas

In [ ]:
df_periodo_seleccionado.loc[condicion_train,"split"] = "train"
df_periodo_seleccionado.loc[condicion_validacion,"split"] = "valid"
df_periodo_seleccionado.loc[condicion_test,"split"] = "test"

### 6.2 - Comprobación de Etiquetas

Se debe tener certeza de que ninguna fecha se quede sin etiquetas

In [ ]:
print("Todos los días del periodo:")
display(df_periodo_seleccionado["split"].value_counts(dropna=False))

print("\nDías elegibles por conjunto:")
resumen_conjuntos = (
    df_periodo_seleccionado
    .loc[df_periodo_seleccionado["Elegible"]]
    .groupby("split")
    .size()
)

display(resumen_conjuntos)

assert df_periodo_seleccionado["split"].isna().sum() == 0, (
    "Existen fechas sin asignación train/valid/test."
)

Todos los días del periodo:


,count
split,
train,2922
valid,731
test,730



Días elegibles por conjunto:


,0
split,
test,506
train,1657
valid,439


## 7 - Caracterización de la precipitación en el conjunto experimental

---

Teniendo en cuenta que los registros de precipitación diaria puenden tener valores iguales a cero o positivos. Antes de generar la máscara artificial conviene distinguir:

```text
día seco      → P_obj = 0 mm
día lluvioso  → P_obj > 0 mm
```

Al crear estas variables, ya que provienen de la misma información registrada en la estación objetivo **no será una feature de los modelos**, por el contrario, el uso de estas es exclusivamente experimental:

- comprobar la composición de train, valid y test;
- construir una máscara que no seleccione accidentalmente casi solo días de una misma condición;
- permitir posteriormente reportar métricas separadas para días secos y lluviosos si fuese necesario.

### Precaución

Cabe aclarar que cuando el valor registrado es `NaN natural` no se puede clasificar como día seco. Por ello la columna se crea como booleana anulable y solo se completa donde la estación objetivo tiene una observación real.

In [ ]:
precipitacion_obj = df_periodo_seleccionado[ESTACION_OBJETIVO]

df_periodo_seleccionado["es_lluvioso"] = (
    (precipitacion_obj > 0)
    .where(precipitacion_obj.notna())
    .astype("boolean")
)

### 7.1 - Resumen Dia seco / lluvioso

In [ ]:
df_elegibles = df_periodo_seleccionado.loc[
    df_periodo_seleccionado["Elegible"]
].copy()

resumen_lluvia_split = (
    df_elegibles
    .groupby(["split", "es_lluvioso"])
    .size()
     )

resumen_lluvia_split

split  es_lluvioso
test   False            66
       True            440
train  False           277
       True           1380
valid  False            81
       True            358
dtype: int64

Para organizar la tabla anterior, se usa el método `unstack()`

In [ ]:
resumen_lluvia_split = resumen_lluvia_split.unstack(level='es_lluvioso', fill_value=0).rename(columns={
        False: "dias_secos",
        True: "dias_lluviosos"
    })

In [ ]:
resumen_lluvia_split

es_lluvioso,dias_secos,dias_lluviosos
split,,
test,66,440
train,277,1380
valid,81,358


In [ ]:
resumen_lluvia_split["dias_totales"] = (
    resumen_lluvia_split["dias_secos"]
    + resumen_lluvia_split["dias_lluviosos"]
)

resumen_lluvia_split["pct_secos"] = (
    resumen_lluvia_split["dias_secos"]
    / resumen_lluvia_split["dias_totales"]
    * 100
).round(2)

resumen_lluvia_split["pct_lluviosos"] = (
    resumen_lluvia_split["dias_lluviosos"]
    / resumen_lluvia_split["dias_totales"]
    * 100
).round(2)

resumen_lluvia_split

es_lluvioso,dias_secos,dias_lluviosos,dias_totales,pct_secos,pct_lluviosos
split,,,,,
test,66,440,506,13.04,86.96
train,277,1380,1657,16.72,83.28
valid,81,358,439,18.45,81.55


La distribución de días secos y lluviosos se conserva en cada bloque, respetando la fragmentación cronológica que hay en cada grupo.

Lo que se hará en la simulación es seleccionar aproximadamente la misma **fracción** dentro de cada estrato seco/lluvioso, evitando que una selección puramente aleatoria pueda dejar accidentalmente una condición insuficientemente representada.



## 8 - Simulación controlada de datos faltantes

---




### 8.1 - ¿Qué se pretende simular?

Como ya se ha mencionado, como los registros `NaN` naturales no tienen un valor real conocido, estos no sirven para calcular directamente métricas como MAE o RMSE.

La evaluación se realizará tomando observaciones reales conocidas y ocultando artificialmente una parte de ellas:

```text
valor real conocido
        ↓
seleccionado por la máscara
        ↓
se guarda y_true
        ↓
se sustituye temporalmente por NaN
        ↓
Notebook 04 genera una imputación
        ↓
predicción vs y_true
```

### ¿Dónde se ocultarán valores?

- **Train:** no se oculta la variable objetivo para el entrenamiento. ya que se necesitan las observaciones conocidas para aprender la relación entre la precipitación objetivo y sus predictores.
- **Valid:** se oculta una fracción reproducible de los días elegibles.
- **Test:** se oculta una fracción reproducible de los días elegibles.

### Patrón principal de faltantes

Para mantener el alcance controlado se utilizará un patrón **puntual aleatorio estratificado por seco/lluvioso**.

### 8.2 - Porcentaje de ocultamiento

Se fija inicialmente:

\[
FRAC\_MASK = 0.25
\]

es decir, aproximadamente el **25 %** de las observaciones elegibles de cada estrato dentro de `valid` y `test`.

Esta proporción no se optimiza como un hiperparámetro. Se utiliza como una decisión experimental fija que proporciona un número suficiente de casos para evaluación y, al mismo tiempo, mantiene una simulación parcial de faltantes dentro de cada periodo.

La selección se realiza de manera reproducible mediante `RANDOM_STATE = 42`.

In [ ]:
# Semilla para reproducibilidad
RANDOM_STATE = 42

# Fracción de observaciones elegibles que se ocultará
# dentro de VALID y TEST.
FRAC_MASK = 0.25

### 8.3 - Construcción de la máscara estratificada

El procedimiento será:

```text
para VALID y TEST:
    seleccionar días elegibles

    dentro de cada split separar:
        días secos
        días lluviosos

    tomar 25 % de cada grupo

    marcar esas fechas:
        mask_simulada = True
```

La variable `mask_simulada` únicamente indica **qué observaciones conocidas serán ocultadas**. En esta celda todavía no se modifica la precipitación.

In [ ]:

df_periodo_seleccionado["mask_simulada"] = False

# en candidatos se almacena unicamente registros con las etiquetas de validación y test
candidatos = df_periodo_seleccionado.loc[
    df_periodo_seleccionado["Elegible"]
    & df_periodo_seleccionado["split"].isin(["valid", "test"])
]

# Seleccionar aleatoriamente el 25 % de cada grupo para ocultar - (solamente selección)
indices_ocultar = (
    candidatos
    .groupby(["split", "es_lluvioso"])
    .sample(
        frac=FRAC_MASK,
        random_state=RANDOM_STATE
    )
    .index
)

# Marcar las observaciones seleccionadas
df_periodo_seleccionado.loc[
    indices_ocultar,
    "mask_simulada"
] = True

### 8.4 - Comprobación de la máscara creada

No se quiere tener la misma cantidad de días secos y lluviosos, sino aplicar  la misma **fracción de ocultamiento** a cada estrato. A continuación se resumen los valores.

In [ ]:
resumen_fraccion_mask = []

for split_actual in ["valid", "test"]:
    for condicion_lluvia in [False, True]:

        candidatos = (
            df_periodo_seleccionado["Elegible"]
            &
            (df_periodo_seleccionado["split"] == split_actual)
            &
            (df_periodo_seleccionado["es_lluvioso"] == condicion_lluvia)
        )

        ocultados = candidatos & df_periodo_seleccionado["mask_simulada"]

        n_candidatos = candidatos.sum()
        n_ocultados = ocultados.sum()

        resumen_fraccion_mask.append({
            "split": split_actual,
            "es_lluvioso": condicion_lluvia,
            "candidatos": n_candidatos,
            "ocultados": n_ocultados,
            "fraccion_real_pct": (
                round(n_ocultados / n_candidatos * 100, 2)
                if n_candidatos > 0
                else np.nan
            )
        })

df_resumen_fraccion_mask = pd.DataFrame(resumen_fraccion_mask)
df_resumen_fraccion_mask

,split,es_lluvioso,candidatos,ocultados,fraccion_real_pct
0,valid,False,81,20,24.69
1,valid,True,358,90,25.14
2,test,False,66,16,24.24
3,test,True,440,110,25.00


### 8.5 - Conservación del valor real (`ground truth`)

Antes de sustituir cualquier valor por `NaN`, se guarda una copia de la precipitación original de la estación objetivo.

Se utilizarán dos columnas diferentes:

- `P_obj_original`: conserva la información tal como salió del Notebook 02;
- `P_obj_modelo`: será la versión que utilizará posteriormente el procedimiento de imputación.

Esta separación evita destruir accidentalmente la verdad conocida.

In [ ]:
df_periodo_seleccionado["P_obj_original"] = (
    df_periodo_seleccionado[ESTACION_OBJETIVO]
)

df_periodo_seleccionado["P_obj_modelo"] = (
    df_periodo_seleccionado["P_obj_original"].copy()
)

In [ ]:
df_periodo_seleccionado.head()

CodigoEstacion,21205710,21201230,21201600,es_nan_nat_obj,es_nan_nat_aux1,es_nan_nat_aux2,n_aux_data,Elegible,Completo_3_estaciones,split,es_lluvioso,mask_simulada,P_obj_original,P_obj_modelo
Fecha,,,,,,,,,,,,,,
2004-01-01,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN
2004-01-02,NaN,0.0,NaN,True,False,True,1,False,False,train,<NA>,False,NaN,NaN
2004-01-03,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN
2004-01-04,NaN,NaN,NaN,True,True,True,0,False,False,train,<NA>,False,NaN,NaN
2004-01-05,NaN,2.5,0.0,True,False,False,2,False,False,train,<NA>,False,NaN,NaN


Se construye además una tabla independiente con la verdad de las observaciones artificialmente ocultadas.

Esta tabla se utilizará **solo para evaluación**, después de obtener las predicciones.

In [ ]:
info_y_verdadera = (
    df_periodo_seleccionado.loc[
            df_periodo_seleccionado["mask_simulada"],
        [
            "split",
            "P_obj_original",
            "es_lluvioso",
            "n_aux_data",
            "Completo_3_estaciones"
        ]
    ]
    .rename(columns={"P_obj_original": "y_true"})
    .copy()
)

info_y_verdadera.index.name = "Fecha"

info_y_verdadera.head()

CodigoEstacion,split,y_true,es_lluvioso,n_aux_data,Completo_3_estaciones
Fecha,,,,,
2012-01-24,valid,0.1,True,2,True
2012-02-13,valid,6.8,True,2,True
2012-02-16,valid,0.1,True,1,False
2012-02-17,valid,0.0,False,1,False
2012-02-18,valid,3.6,True,2,True


La tabla info_y_verdadera conserva únicamente la información necesaria para evaluar posteriormente las imputaciones realizadas sobre los valores ocultados artificialmente. Principalmente almacena el valor real de precipitación (y_true), el conjunto temporal al que pertenece cada observación y la cantidad de información auxiliar disponible. Con esta info, el proceso de imputación puede trabajar con los valores ocultos sin perder la referencia original necesaria para calcular las métricas de desempeño.

### 8.6 - Aplicación del faltante artificial

Ahora sí se ocultan los valores seleccionados, pero **únicamente en `P_obj_modelo`**.

La columna original permanece intacta y la tabla `info_y_verdadera` conserva los valores reales destinados a la evaluación.

In [ ]:
df_periodo_seleccionado.loc[
    df_periodo_seleccionado["mask_simulada"],
    "P_obj_modelo"
] = np.nan

### 8.7 - Verificaciones de integridad de la máscara

Antes de continuar con la construcción de features se realizan varias comprobaciones automáticas mediante [`assert`](https://www.datacamp.com/tutorial/understanding-the-python-assert-statement?utm_aid=157296744657&utm_loc=9197744-&utm_mtd=-c&utm_kw=).

`assert` significa:

> esta condición debe ser verdadera; si no lo es, detén la ejecución.

Las comprobaciones garantizan que:

1. no se ocultó ningún `NaN` natural;
2. todos los valores ocultados pertenecían a observaciones elegibles;
3. no se ocultaron valores del conjunto de entrenamiento;
4. todos los valores guardados en `ground_truth` son conocidos;
5. todos los puntos de la máscara aparecen como `NaN` en `P_obj_modelo`;
6. el valor original se mantiene intacto.

#### 1- No se ocultó ningún `NaN` natural


In [ ]:
errores_nan_natural = (
    df_periodo_seleccionado["mask_simulada"]
    &
    df_periodo_seleccionado["es_nan_nat_obj"]
).sum()

assert errores_nan_natural == 0, \
    "Se ocultó artificialmente un NaN natural."

#### 2- Todos los valores ocultados pertenecían a observaciones elegibles;


In [ ]:
errores_no_elegibles = (
    df_periodo_seleccionado["mask_simulada"]
    &
    ~df_periodo_seleccionado["Elegible"]
).sum()

assert errores_no_elegibles == 0, \
    "Existen valores simulados fuera del conjunto elegible."

#### 3- No se ocultaron valores del conjunto de entrenamiento;


In [ ]:
errores_train = (
    df_periodo_seleccionado["mask_simulada"]
    &
    (df_periodo_seleccionado["split"] == "train")
).sum()

assert errores_train == 0, \
    "Se ocultaron valores artificiales en train."

#### 4- Todos los valores guardados en `ground_truth` son conocidos;


In [ ]:
errores_y_true = info_y_verdadera["y_true"].isna().sum()

assert errores_y_true == 0, \
    "La información verdadera contiene valores desconocidos."

#### 5- Todos los puntos de la máscara aparecen como `NaN` en `P_obj_modelo`;


In [ ]:
errores_mascara = (
    df_periodo_seleccionado.loc[
        df_periodo_seleccionado["mask_simulada"],
        "P_obj_modelo"
    ]
    .notna()
    .sum()
)

assert errores_mascara == 0, \
    "La máscara no se aplicó correctamente."

#### 6- El valor original se mantiene intacto.

In [ ]:
serie_original_igual = (
    df_periodo_seleccionado["P_obj_original"]
    .equals(
        df_periodo_seleccionado[ESTACION_OBJETIVO]
    )
)

assert serie_original_igual, \
    "La serie original fue modificada."

## 9 - Salida del Notebook 3

In [ ]:

import json

# Carpeta de salida
RUTA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

# 1. Dataset experimental completo
df_periodo_seleccionado.to_parquet(
    RUTA_PROCESSED / "03_dataset_experimental.parquet"
)

# 2. Valores verdaderos de las observaciones ocultadas
info_y_verdadera.to_parquet(
    RUTA_PROCESSED / "03_info_y_verdadera.parquet"
)

# 3. Configuración del experimento
config_experimento = {
    "estacion_objetivo": ESTACION_OBJETIVO,
    "estaciones_auxiliares": ESTACIONES_AUXILIARES,
    "train_inicio": str(TRAIN_INICIO),
    "train_fin": str(TRAIN_FIN),
    "valid_inicio": str(VALID_INICIO),
    "valid_fin": str(VALID_FIN),
    "test_inicio": str(TEST_INICIO),
    "test_fin": str(TEST_FIN),
    "frac_mask": FRAC_MASK,
    "random_state": RANDOM_STATE
}

with open(
    RUTA_PROCESSED / "03_config_experimento.json",
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        config_experimento,
        archivo,
        indent=4
    )

print("✓ Salidas del Notebook 03 guardadas correctamente.")

✓ Salidas del Notebook 03 guardadas correctamente.


### Archivos de salida del Notebook 03

#### `03_dataset_experimental.parquet`

Contiene el conjunto de datos experimental que será utilizado en el **Notebook 04** para el modelado y la evaluación.

Incluye las precipitaciones de la estación objetivo y de las estaciones auxiliares, junto con las variables de control generadas durante el diseño experimental, entre ellas:

- `split`: identifica si cada observación pertenece a `train`, `valid` o `test`.
- `Elegible`: indica si la fecha cumple los criterios definidos para participar en el experimento.
- `mask_simulada`: identifica las observaciones que fueron ocultadas artificialmente.
- `P_obj_original`: conserva el valor original de precipitación de la estación objetivo.
- `P_obj_modelo`: contiene la serie utilizada durante el proceso de modelado, incluyendo los valores ocultados artificialmente como `NaN`.
- `n_aux_data`: indica cuántas estaciones auxiliares tienen información disponible en cada fecha.

También se conservan las demás variables de control generadas durante la construcción del experimento.

---

#### `03_info_y_verdadera.parquet`

Contiene únicamente las observaciones que fueron ocultadas artificialmente en los conjuntos de validación y prueba.

Para cada fecha se conserva el valor real de precipitación en la variable `y_true`, junto con información auxiliar necesaria para la evaluación.

Este archivo permite comparar posteriormente las estimaciones realizadas por los métodos de imputación con los valores reales conocidos y calcular métricas de desempeño como **MAE** y **RMSE**.

---

#### `03_config_experimento.json`

Guarda los principales parámetros y decisiones utilizadas para construir el experimento, de manera que puedan ser reutilizados en el **Notebook 04**.

Incluye:

- código de la estación objetivo;
- códigos de las estaciones auxiliares;
- fechas correspondientes a los conjuntos `train`, `valid` y `test`;
- fracción de observaciones ocultadas artificialmente (`frac_mask`);
- semilla aleatoria utilizada para garantizar la reproducibilidad del experimento (`random_state`).

Este archivo permite mantener la configuración del experimento separada del dataset y facilita la reproducibilidad del flujo de trabajo.

En este notebook se construyó el diseño experimental que permitirá evaluar de forma controlada los diferentes métodos de imputación de precipitación diaria.

A partir de las series seleccionadas en el Notebook 02 se identificaron las observaciones elegibles, definidas como aquellas fechas en las que la precipitación de la estación objetivo es conocida y existe información en al menos una estación auxiliar.

Posteriormente, el periodo 2004–2015 se dividió cronológicamente en conjuntos de entrenamiento, validación y prueba. Sobre los conjuntos de validación y prueba se generó una máscara reproducible de datos faltantes artificiales, conservando de manera independiente los valores reales de precipitación utilizados posteriormente como referencia de evaluación.

Las comprobaciones realizadas confirmaron que los valores ocultados corresponden exclusivamente a observaciones originalmente conocidas y elegibles, que el conjunto de entrenamiento permanece sin faltantes artificiales y que la serie original no fue modificada.

Como resultado, el Notebook 03 genera un conjunto experimental reproducible que contiene la estructura temporal, las observaciones naturales, los faltantes simulados y los valores verdaderos necesarios para la evaluación.

El Notebook 04 utilizará estas salidas para implementar los métodos tradicionales y de Machine Learning, construir las variables predictoras y comparar el desempeño de las imputaciones mediante métricas de error.